# 02 · Quality Control — Human Femoral Head Visium HD

**Input:** `results/adata/01_raw.h5ad`  
**Outputs:** QC metrics, figures, filtering report, and `results/adata/02_qc.h5ad`.

This dataset contains **56 µm aggregated bins**, not individual cells. We inspect distributions before choosing thresholds. No image overlay is attempted because full-resolution coordinates have not yet been registered to the hires PNG. This notebook can run independently, including under Papermill.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse
from importlib.metadata import version

# Works from either the repository root or notebooks/.
HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / "data").is_dir() and (p / "results").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Run from visium_hd_tutorial/ or its notebooks/ directory")
INPUT = ROOT / "results/adata/01_raw.h5ad"
OUT = ROOT / "results/adata/02_qc.h5ad"
FIG = ROOT / "results/figures"
TAB = ROOT / "results/tables"
for d in (OUT.parent, FIG, TAB): d.mkdir(parents=True, exist_ok=True)
assert INPUT.is_file(), f"Missing input: {INPUT}. Run 01_data_loading first."
print("Python:", sys.executable)
print("Scanpy:", version("scanpy"))
print("Project:", ROOT)


## 1 · Load and validate raw counts

Check that the handoff artifact is intact and still contains sparse, nonnegative raw counts and spatial coordinates.

In [ ]:
adata = sc.read_h5ad(INPUT)
assert sparse.issparse(adata.X), "Expected sparse raw counts"
assert adata.obs_names.is_unique and adata.var_names.is_unique
assert "spatial" in adata.obsm and adata.obsm["spatial"].shape == (adata.n_obs, 2)
assert "in_tissue" in adata.obs
assert np.isfinite(adata.obsm["spatial"]).all()
assert adata.X.data.size == 0 or np.isfinite(adata.X.data).all()
assert adata.X.data.size == 0 or (adata.X.data >= 0).all()
print(adata)
print("Tissue flags:", adata.obs["in_tissue"].value_counts(dropna=False).to_dict())


## 2 · Calculate per-bin QC metrics

Mitochondrial gene names are detected using the human `MT-` prefix. If none are found, mitochondrial percentage is reported as unavailable rather than silently set to zero.

In [ ]:
adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
n_mt = int(adata.var["mt"].sum())
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"] if n_mt else [], percent_top=None, inplace=True, log1p=False)
if not n_mt:
    adata.obs["pct_counts_mt"] = np.nan
print("Mitochondrial genes detected:", n_mt)
cols = ["total_counts", "n_genes_by_counts", "pct_counts_mt"]
print(adata.obs[cols].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).round(2).to_string())


## 3 · QC visualizations

Distributions and count–gene relationship help identify candidate thresholds. Spatial scatter plots use **full-resolution coordinate units only**, without overlaying the hires image.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, label in zip(axes, cols, ["Total UMI counts / bin", "Detected genes / bin", "Mitochondrial counts (%)"]):
    vals = adata.obs[col].dropna().to_numpy()
    if len(vals): ax.hist(vals, bins=60)
    else: ax.text(.5,.5,"Not available",ha="center",va="center",transform=ax.transAxes)
    ax.set_xlabel(label); ax.set_ylabel("Bins")
fig.tight_layout()
fig.savefig(FIG / "02_qc_distributions.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(adata.obs["total_counts"], adata.obs["n_genes_by_counts"], s=4, alpha=.3)
ax.set(xlabel="Total UMI counts", ylabel="Detected genes", title="Counts vs genes")
fig.tight_layout()
fig.savefig(FIG / "02_qc_counts_vs_genes.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
xy = adata.obsm["spatial"]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col in zip(axes, ["total_counts", "n_genes_by_counts"]):
    dots = ax.scatter(xy[:, 0], xy[:, 1], c=np.log1p(adata.obs[col].to_numpy()), s=5, cmap="viridis", rasterized=True)
    ax.invert_yaxis(); ax.set_aspect("equal", adjustable="box")
    ax.set(xlabel="Fullres pixel X", ylabel="Fullres pixel Y", title=f"log1p({col})")
    fig.colorbar(dots, ax=ax, shrink=.8)
fig.tight_layout()
fig.savefig(FIG / "02_qc_spatial_coordinates_only.png", dpi=180, bbox_inches="tight")
plt.show()


## 4 · Filtering configuration

**Default is diagnostic-only** (`APPLY_FILTERS = False`). This preserves all bins until QC thresholds have been reviewed with the team. To filter in a later run, set `APPLY_FILTERS = True` and choose thresholds based on the figures and quantiles above. `in_tissue` is kept as metadata; optional tissue-only filtering can be enabled separately. Filtering is never applied to genes in this notebook.

In [ ]:
# Adjustable QC policy — intentionally no filtering by default.
APPLY_FILTERS = False
MIN_COUNTS = 100
MIN_GENES = 50
MAX_PCT_MT = 25.0
REQUIRE_IN_TISSUE = False

keep = np.ones(adata.n_obs, dtype=bool)
if APPLY_FILTERS:
    keep &= adata.obs["total_counts"].to_numpy() >= MIN_COUNTS
    keep &= adata.obs["n_genes_by_counts"].to_numpy() >= MIN_GENES
    if n_mt:
        keep &= adata.obs["pct_counts_mt"].to_numpy() <= MAX_PCT_MT
    if REQUIRE_IN_TISSUE:
        keep &= adata.obs["in_tissue"].to_numpy() == 1

report = pd.DataFrame([{
    "input_bins": adata.n_obs,
    "retained_bins": int(keep.sum()),
    "removed_bins": int((~keep).sum()),
    "apply_filters": APPLY_FILTERS,
    "min_counts": MIN_COUNTS if APPLY_FILTERS else "not_applied",
    "min_genes": MIN_GENES if APPLY_FILTERS else "not_applied",
    "max_pct_mt": MAX_PCT_MT if APPLY_FILTERS and n_mt else "not_applied",
    "require_in_tissue": REQUIRE_IN_TISSUE if APPLY_FILTERS else "not_applied",
    "mitochondrial_genes_found": n_mt,
}])
print(report.to_string(index=False))
assert keep.any(), "No bins remain; review thresholds"
adata_qc = adata[keep].copy()


## 5 · Save outputs and verify reload

The output contains **raw, unnormalized counts**, augmented with QC metrics. Normalization happens only in Notebook 03.

In [ ]:
adata.obs[cols + ["in_tissue"]].to_csv(TAB / "02_qc_metrics_all_bins.csv", index_label="barcode")
report.to_csv(TAB / "02_qc_filter_report.csv", index=False)
adata_qc.uns["qc_filter_policy"] = {
    "applied": bool(APPLY_FILTERS),
    "min_counts": int(MIN_COUNTS),
    "min_genes": int(MIN_GENES),
    "max_pct_mt": float(MAX_PCT_MT),
    "require_in_tissue": bool(REQUIRE_IN_TISSUE),
    "mitochondrial_genes_found": int(n_mt),
}
adata_qc.write_h5ad(OUT)
check = sc.read_h5ad(OUT)
assert check.shape == adata_qc.shape
assert check.obs_names.equals(adata_qc.obs_names)
assert np.array_equal(check.obsm["spatial"], adata_qc.obsm["spatial"])
assert sparse.issparse(check.X)
assert "total_counts" in check.obs and "n_genes_by_counts" in check.obs
print("PASS — 02_quality_control completed")
print("Output:", OUT.relative_to(ROOT))
print("AnnData:", check.shape)
print("Filters applied:", APPLY_FILTERS)


### Next: `03_normalization.ipynb`

Read `results/adata/02_qc.h5ad`; preserve raw counts before normalization, then perform normalization, log transformation, and feature selection. **Review QC figures before enabling any filtering.**